# Advanced OOP – Part 2 Practice Notebook


Topics covered in this session:  
- abstract classes and abstract methods  
- multiple inheritance  
- ambiguity and MRO  
- the diamond problem  
- `super()` and `**kwargs`  
- the main guard

**Note:** Try to solve each task yourself first. Test small pieces of code, inspect objects, and read errors carefully. In this notebook especially, the errors are often part of the exercises.

## Abstract Classes: designing a shared blueprint

When several classes clearly belong to the same family, it is often useful to create a common base class.  
But sometimes that base class is only a **blueprint**: it describes what all subclasses must have, without being something we should create directly.

In the next exercise, imagine you are helping a small media lab that works with different kinds of content creators.

### Formative Exercise 1

A digital media lab wants to track different kinds of creators. Every creator should have:

- a `name`
- a `platform` (for example: `"YouTube"`, `"Instagram"`, `"Podcast"` ,`"Spotify"`)

However, a generic `Creator` is too vague to be useful on its own. The lab only wants concrete creator types, such as travel writers or podcast hosts.

Please complete the code below so that:

1. `Creator` is an **abstract class**
2. `Creator` stores `name` and `platform`
3. `Creator` defines an abstract method called `create_content()`
4. it should **not** be possible to instantiate `Creator` directly

After implementing it, test and check that Python prevents direct instantiation.

**Design question:** Why is an abstract class a better fit here than a normal class with a vague default method?

In [18]:

# Exercise 1
from abc import ABC, abstractmethod

class Creator(ABC):
     def __init__(self, name, platform, **kwargs) -> None:
         self.name = name
         self.platform = platform
         super().__init__(**kwargs)
     @abstractmethod
     def create_content():
         pass
     



### Formative Exercise 2

Now create two concrete subclasses (`TravelWriter`, `PodcastHost`), inheriting from the abstract class `Creator`:

##### 1. `TravelWriter` should include:
- extra attribute: `region`
- method `write_article()` returns  
  `"<name> writes a travel article about <region>."`
- method `create_content()` returns  
  `"Travel article in progress."`

##### 2. `PodcastHost` should include:
- extra attribute: `topic`
- method `record_episode()` returns  
  `"<name> records a podcast episode about <topic>."`
- method `create_content()` returns  
  `"Podcast episode in progress."`

For this exercise, **do not use `super()`** yet. Instead, call the parent constructor explicitly. After that, create one object of each class and test all methods.

**Why this matters:** before we use `super()`, it is useful to first see inheritance in a more direct and mechanical way.

In [19]:

# Exercise 2

# Test your classes below
class TravelWriter(Creator):
    def __init__(self,region, **kwargs) -> None:
        super().__init__(**kwargs)
        self.region = region
    
    def create_content(self):
        return "Travel article in progress"
    
    def write_article(self):
        return f"{self.name} writes a travel article about {self.region}"
    
class PodcastHost(Creator):
    def __init__(self, topic, **kwargs) -> None:
        super().__init__(**kwargs)
        self.topic = topic
    
    def create_content(self):
        return "Podcast episode in progress"
    
    def record_episode(self):
        return f"{self.name} records a podcast episode about {self.topic}"





---

## Multiple Inheritance: one object, several roles

In real life, people sometimes combine responsibilities. A person can be both a writer and a host, or both a student and a developer, or both a designer and a manager.

Multiple inheritance lets us model that kind of situation — but it also creates new questions:
- which parent method should be used?
- which constructor runs first?
- what happens when two parents share the same base class?

### Formative Exercise 3

Create a class called `FieldReporter` that inherits from **both** `TravelWriter` and `PodcastHost`.

For now, only implement:

- `create_content()` returning `"Mixed field report in progress."`

Do **not** write a constructor yet.

Then answer these questions before running much code:

1. Which methods should a `FieldReporter` object already have access to?
2. Will it automatically have both `write_article()` and `record_episode()`?
3. Why might construction become tricky later?

After that, define the class in code.

### Formative Exercise 4

Read the code below before you run it.

```python
class CameraCreator:
    def publish(self):
        print("Photo story published")

class AudioCreator:
    def publish(self):
        print("Audio story published")

class HybridCreator(AudioCreator, CameraCreator):
    pass

h = HybridCreator()
h.publish()
```
.......

Answer the following:

1. What will be printed?
2. Why does Python choose that version?
3. Would the answer change if `HybridCreator(CameraCreator, AudioCreator)` were used instead?

Then run your own version of the code to check your reasoning.

In [6]:

# Exercise 4
class CameraCreator:
    def publish(self):
        print("Photo story published")

class AudioCreator:
    def publish(self):
        print("Audio story published")

class HybridCreator(AudioCreator, CameraCreator):
    pass

h = HybridCreator()
h.publish()


Audio story published


### Formative Exercise 5

Using either `HybridCreator` from the previous task or `FieldReporter`, inspect the **method resolution order**.

Your tasks:

1. print the MRO using `.mro()`
2. identify the order in which Python searches parent classes
3. explain in one or two sentences why MRO matters in multiple inheritance

**Important idea:** MRO tells Python where to **look first**. 
It does **not** mean every class in the list automatically runs its code.

In [7]:

# Exercise 5
FieldReporter.mro()


[__main__.FieldReporter,
 __main__.TravelWriter,
 __main__.PodcastHost,
 __main__.Creator,
 abc.ABC,
 object]

### Formative Exercise 6

Now comes the interesting part.

`TravelWriter` expects:
- `region`
- `name` 
- `platform`

`PodcastHost` expects:
- `topic`
- `name`
- `platform`


But `FieldReporter` inherits from both.

Try something like this:

```python
reporter = FieldReporter( "Iceland", "Mina", "YouTube")
```

and then inspect what happens.

Your tasks:

1. try to instantiate `FieldReporter`
2. observe the error or behavior carefully
3. explain:
   - which constructor Python tries first
   - why that happens
   - why the object may end up missing data from one parent

After that, try calling `record_episode()` or checking whether `topic` exists.

In [8]:

# Exercise 6
reporter = FieldReporter( "Iceland", "Mina", "YouTube")


TypeError: __init__() missing 1 required positional argument: 'topic'

### Formative Exercise 7

Fix `FieldReporter` in the most direct way first.

Write a constructor for `FieldReporter` so that it receives:

- `name`
- `platform`
- `region`
- `topic`

and then manually calls **both** parent constructors.

After that:

1. create a `FieldReporter`
2. show that both `region` and `topic` now exist
3. call both `write_article()` and `record_episode()`

**Food for thought:** if both parent constructors call the `Creator` constructor, what might happen behind the scenes?

In [27]:
# Exercise 7
class FieldReporter2(TravelWriter, PodcastHost):
    def __init__(self, name, platform, region, topic) -> None:
        super().__init__(
            name=name,
            platform=platform,
            region= region,
            topic = topic,
            
            
        )
    def create_content(self):
        return "Mixed field report in progress"

reporter = FieldReporter2( "Iceland", "Mina", "YouTube", "Cheese")
print(FieldReporter2.mro())
print(reporter.write_article())
print(reporter.record_episode())



[<class '__main__.FieldReporter2'>, <class '__main__.TravelWriter'>, <class '__main__.PodcastHost'>, <class '__main__.Creator'>, <class 'abc.ABC'>, <class 'object'>]
Iceland writes a travel article about YouTube
Iceland records a podcast episode about Cheese


### Formative Exercise 8

The manual fix above works, but it hides a deeper issue.

Suppose `Creator.__init__()`  ––the parent class––prints a message such as:

```python
print("Creator initialized")
```

Now think about what happens when `FieldReporter.__init__()` manually calls both parent constructors.

Your tasks:

1. temporarily modify `Creator.__init__()` so it prints when it runs
2. create a `FieldReporter`
3. observe how many times the base constructor is called
4. explain in plain language why this is known as the **diamond problem**

**Note:** duplicate initialization can mean repeated setup, repeated connections, repeated file access, or repeated side effects.

In [ ]:
# Exercise 8

### Formative Exercise 9

Now refactor the design using the safer pattern from class:

- use `super()`
- use `**kwargs`
- let each class take what it needs and pass the rest along

Refactor `Creator`, `TravelWriter`, `PodcastHost`, and `FieldReporter` so that:

1. `Creator` still stores `name` and `platform`
2. `TravelWriter` takes `region`
3. `PodcastHost` takes `topic`
4. `FieldReporter` uses **one** `super().__init__(...)` call
5. the shared base initialization happens only once

After refactoring, create one object and test that everything still works.

**Why this pattern is useful:** it cooperates with the MRO instead of fighting it.

In [ ]:
# Exercise 9

---

## Main Guard

### Formative Exercise 10

One final software-engineering habit.

Suppose `main.py` contains test code that creates objects and prints output.  
What happens if another file imports `main.py`?

Your tasks:

1. write a small `main()` function that creates and tests a `FieldReporter`
2. protect the execution using a **main guard**
3. explain what problem this solves

**Key idea:** importing a file should usually give access to its definitions, not automatically run all of its test code.

In [ ]:

# Exercise 10

---

## Final Reflection

Answer these in your own words after finishing the notebook:

- What is the practical difference between a normal base class and an abstract class?
- Why can multiple inheritance become confusing without MRO?
- What problem does `super()` solve in a diamond-shaped hierarchy?
- Why is the main guard such a small line of code with such a useful effect?